In [ ]:
%cd /content
# Ensure CWD is valid (not inside a deleted directory)
# @title Clean install — run once, then restart runtime
import shutil, os, sys
# 1. Remove stale __pycache__
for path in ['DeFT', 'deft']:
    if os.path.exists(path):
        for root, dirs, files in os.walk(path):
            if '__pycache__' in dirs:
                shutil.rmtree(os.path.join(root, '__pycache__'))
# 2. Remove cached modules
for mod in list(sys.modules.keys()):
    if mod.startswith('deft'):
        del sys.modules[mod]
# 3. Delete old repo and re-clone
shutil.rmtree('DeFT', ignore_errors=True)
!git clone https://github.com/Xiao-Zhanpeng/DeFT.git
print('Clean install done. Go to Runtime → Restart runtime before running rest of notebook.')


# DeFT: Medical Image Denoising Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xiao-Zhanpeng/DeFT/blob/main/demo.ipynb)

**Descriptor-Forked Test-Time Adaptation** — source-free, single-image denoising.

1. Set runtime: `Runtime` → `Change runtime type` → `T4 GPU`
2. Run all cells: `Runtime` → `Run all`
3. See denoised results on three example medical images (CT / MRI / X-ray)

In [ ]:
# @title Clone repository and install dependencies
%cd DeFT
!pip install -q torch numpy scipy scikit-image monai tqdm matplotlib
!pip install -q huggingface_hub
import torch
import numpy as np
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title Download pretrained checkpoint (Hugging Face)
!mkdir -p checkpoints
from huggingface_hub import hf_hub_download
ckpt_path = hf_hub_download(
    repo_id="Lockbro/deft-checkpoints",
    filename="unet_source_checkpoint.pt",
    local_dir="checkpoints"
)
print(f"Checkpoint: {ckpt_path}")

In [ ]:
# @title Load DeFT model with pretrained weights
from deft import DeFT, DeFTBackbone

backbone = DeFTBackbone.from_pretrained("checkpoints/unet_source_checkpoint.pt")
model = DeFT(denoiser=backbone)
model.cuda()
print(f"Model ready. Trainable adapters: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# @title Run DeFT on example images (CT / MRI / X-ray)
import matplotlib.pyplot as plt
from pathlib import Path

samples = [
    ("Q₁ Mayo CT",        "examples/q1_mayo_ct_noisy.npy",     5),
    ("Q₂ fastMRI Knee",   "examples/q2_fastmri_knee_noisy.npy", 5),
    ("Q₃ Chest X-ray",    "examples/q3_chest_xray_noisy.npy",   5),
]

fig, axes = plt.subplots(len(samples), 2, figsize=(10, 12))

for i, (name, path, steps) in enumerate(samples):
    img = np.load(path).astype(np.float32)
    if img.ndim == 2:
        img = img[np.newaxis, ...]
    noisy = torch.from_numpy(img).float().cuda().unsqueeze(0)

    print(f"Adapting {name} ({steps} steps) ...", end=" ")
    denoised = model.adapt(noisy, steps=steps)
    print("done")

    axes[i][0].imshow(noisy.squeeze().cpu().numpy(), cmap="gray")
    axes[i][0].set_title(f"{name} — Noisy", fontsize=12)
    axes[i][0].axis("off")
    axes[i][1].imshow(denoised.squeeze().cpu().numpy(), cmap="gray")
    axes[i][1].set_title(f"{name} — DeFT Denoised", fontsize=12)
    axes[i][1].axis("off")

plt.tight_layout()
plt.suptitle("DeFT: Source-Free Single-Image Test-Time Adaptation",
             fontsize=14, y=1.02)
plt.show()
print("\nAll three domains processed. Results above.")